In [1]:
import cv2
import numpy as np

CAMERA_INDEX = 0
RATIO        = 0.75   # stricter than 0.75 — fewer but better matches
MAX_FEATURES = 600
MIN_FLOW_PX  = 1      # skip arrows shorter than 1px (removes stationary noise)

sift  = cv2.SIFT_create(nfeatures=MAX_FEATURES)
flann = cv2.FlannBasedMatcher({"algorithm": 1, "trees": 5}, {"checks": 50})

def get_features(img):
    return sift.detectAndCompute(img, None)

def draw_keypoints(img, kps):
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.drawKeypoints(vis, kps, vis, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    return vis

def match_features(d1, d2):
    if d1 is None or d2 is None or len(d1) < 2 or len(d2) < 2:
        return []
    raw = flann.knnMatch(d1, d2, k=2)
    return [m for m, n in raw if len([m, n]) == 2 and m.distance < RATIO * n.distance]

def ransac_filter(kps1, kps2, matches):
    if len(matches) < 8:
        return matches
    pts1 = np.float32([kps1[m.queryIdx].pt for m in matches])
    pts2 = np.float32([kps2[m.trainIdx].pt for m in matches])
    _, mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, 5.0)
    if mask is None:
        return matches
    return [m for m, keep in zip(matches, mask.flatten()) if keep]

def draw_flow(img, kps1, kps2, matches):
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    drawn = 0
    for m in matches:
        x0, y0 = map(int, kps1[m.queryIdx].pt)
        x1, y1 = map(int, kps2[m.trainIdx].pt)
        if abs(x1-x0) < MIN_FLOW_PX and abs(y1-y0) < MIN_FLOW_PX:
            continue                                    # skip near-zero arrows
        cv2.arrowedLine(vis, (x0, y0), (x1, y1), (0, 255, 0), 2, tipLength=0.4)
        cv2.circle(vis, (x0, y0), 3, (0, 0, 255), -1)
        drawn += 1
    cv2.putText(vis, f"Optical Flow  |  {drawn} vectors",
                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)
    return vis

def run():
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print("Could not open camera.")
        return

    prev_gray, prev_kps, prev_desc = None, None, None
    frame_idx = 0
    print("Press Q to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        kps, desc = get_features(gray)

        feat_vis = draw_keypoints(gray, kps)
        cv2.putText(feat_vis, f"Frame {frame_idx}  |  {len(kps)} kpts",
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)
        cv2.imshow("SIFT Features", feat_vis)

        if prev_kps is not None:
            matches = match_features(prev_desc, desc)
            matches = ransac_filter(prev_kps, kps, matches)   # remove outliers

            match_vis = cv2.drawMatches(
                prev_gray, prev_kps, gray, kps, matches[:100], None,
                matchColor=(0, 255, 0),
                flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
            )
            cv2.putText(match_vis, f"{len(matches)} matches",
                        (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)
            cv2.imshow("SIFT Matches", match_vis)

            if matches:
                flow_vis = draw_flow(gray, prev_kps, kps, matches)
                cv2.imshow("Optical Flow", flow_vis)

            print(f"[{frame_idx:05d}]  {len(kps)} kpts  |  {len(matches)} matches")

        prev_gray, prev_kps, prev_desc = gray, kps, desc
        frame_idx += 1

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run()

Press Q to quit.
[00001]  343 kpts  |  94 matches
[00002]  351 kpts  |  165 matches
[00003]  338 kpts  |  89 matches
[00004]  332 kpts  |  173 matches
[00005]  314 kpts  |  179 matches
[00006]  321 kpts  |  170 matches
[00007]  324 kpts  |  157 matches
[00008]  319 kpts  |  171 matches
[00009]  352 kpts  |  184 matches
[00010]  320 kpts  |  188 matches
[00011]  318 kpts  |  168 matches
[00012]  329 kpts  |  178 matches
[00013]  296 kpts  |  117 matches
[00014]  277 kpts  |  148 matches
[00015]  303 kpts  |  142 matches
[00016]  282 kpts  |  128 matches
[00017]  281 kpts  |  110 matches
[00018]  232 kpts  |  53 matches
[00019]  247 kpts  |  87 matches
[00020]  226 kpts  |  83 matches
[00021]  247 kpts  |  91 matches
[00022]  242 kpts  |  108 matches
[00023]  245 kpts  |  97 matches
[00024]  225 kpts  |  92 matches
[00025]  244 kpts  |  59 matches
[00026]  223 kpts  |  101 matches
[00027]  225 kpts  |  103 matches
[00028]  158 kpts  |  62 matches
[00029]  61 kpts  |  12 matches
[00030]  